In [5]:
import os
import sys
sys.path.append("/home/wengang/vanilla/routing_decision")
from tools.analyze import *

In [6]:
""" Experiments on OLMoE """
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
###-------- Basic settings --------####
output_dir = "../../routing_decision_results/demo_olmoe/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
torch.set_default_device("cuda:0") # or "cpu"
torch.set_grad_enabled(False)
####-------- Model settings --------####
model_id = "allenai/OLMoE-1B-7B-0924" # "allenai/OLMoE-1B-7B-0125"
n_layers = 16
n_dim = 2048
n_heads = 16
n_experts = 64
top_k = 8
####-------- Model loading --------####
from customized_models.modeling_olmoe_customized import OlmoeForCausalLM
model = OlmoeForCausalLM.from_pretrained(model_id, attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(model_id)
router_weight_ls = [model.model.layers[i].mlp.gate.weight for i in range(n_layers)]
from dataset.c4_dataset import *
my_c4_dataset = c4_dataset_helper(dataset_len=1000, min_words=40)

Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.26s/it]


len of my_c4_dataset 1000


In [3]:
""" Figures 2a, 2b """

' Figures 2a, 2b '

In [4]:
""" Figures 2c, 3 """

' Figures 2c, 3 '

In [5]:
""" Figures 4 """

' Figures 4 '

In [7]:
""" Figures * (Appendix) """
from dataset.ioi_dataset import *
prompt_dict_ls_ORIG = gen_ioi_prompt(20, tokenizer, "mixed", {"[PLACE]": PLACES, "[OBJECT]": OBJECTS}, NAMES, ABCD_TEMPLATES, None)
prompt_dict_ls_NEW = gen_abc_prompt(tokenizer, ABCD_TEMPLATES, prompt_dict_ls_ORIG)
## should contain "text", "IO_token_id", "S_token_id", and necessary token_pos at least
## find Name Mover, Negative Name Mover
send_info = {"token_pos_ls":[ i["END_token_pos"] for i in prompt_dict_ls_ORIG ]}
recv_info = {"type":"l","token_pos_ls":[ i["END_token_pos"] for i in prompt_dict_ls_ORIG ]}
## Figure *a
path_patching(prompt_dict_ls_ORIG, prompt_dict_ls_NEW, model, tokenizer, send_info, recv_info, output_dir, n_layers, n_heads, bsz=20, demo_now=True)
## Figure *b~g
# decompose_IOI_map_score(prompt_dict_ls_ORIG, prompt_dict_ls_NEW, model, tokenizer, router_weight_ls, output_dir, n_heads, n_experts, bsz=20, demo_now=True)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [01:38<00:00, 98.89s/it]


In [7]:
""" Figures 5 """

' Figures 5 '

In [8]:
# from ioi_dataset import *
# ioi_dataset_olmoe = gen_ioi_prompt(200, tokenizer, "mixed", {"[PLACE]": PLACES, "[OBJECT]": OBJECTS}, NAMES, ABCD_TEMPLATES, None)
# abc_dataset_olmoe = gen_abc_prompt(tokenizer, ABCD_TEMPLATES, ioi_dataset_olmoe)
# prompt_dict_ls_ORIG = ioi_dataset_olmoe # should contain "text", "IO_token_id", "S_token_id", and necessary token_pos at least
# prompt_dict_ls_NEW = abc_dataset_olmoe
# ## find Name Mover, Negative Name Mover
# send_info = {"token_pos_ls":[ i["END_token_pos"] for i in prompt_dict_ls_ORIG ]}
# recv_info = {"type":"l","token_pos_ls":[ i["END_token_pos"] for i in prompt_dict_ls_ORIG ]}
# decompose_IOI_map_score(prompt_dict_ls_ORIG, prompt_dict_ls_NEW, model, tokenizer, router_weight_ls, output_dir, n_heads, n_experts, bsz=10, demo_now=True)

In [ ]:
from tools.misc import activation_patching
send_info = None
recv_info = {"type":"qkv", "token_pos_ls":([13]*1)}
prompt_maryjohnjohn = "When Mary and John went to the store, John gave a drink to"
prompt_davidmiketom = "When David and Mike went to the store, Tom gave a drink to"
# batch_token = tokenizer(prompt_maryjohnjohn, return_tensors="pt", padding=True) # for test
# batch_token = tokenizer(prompt_davidmiketom, return_tensors="pt", padding=True) # for test
prompt_dict_ls_ORIG = [{"text":prompt_maryjohnjohn, "IO_token_id":[6393], "S_token_id":[2516], "S2_token_id":[2516], "END_token_pos":13}]
prompt_dict_ls_NEW = [{"text":prompt_davidmiketom, "IO_token_id":[5119], "S_token_id":[10035], "S2_token_id":[6270], "END_token_pos":13}]
activation_patching(prompt_dict_ls_ORIG, prompt_dict_ls_NEW, model, tokenizer, send_info, recv_info, output_dir, n_layers, n_heads, bsz=1, demo_now=True)

100%|██████████| 1/1 [04:13<00:00, 253.34s/it]


In [4]:
send_info = None
recv_info = {"type":"o", "token_pos_ls":([9]*1)}
prompt_maryjohnjohn = "When Mary and John went to the store, John gave a drink to"
prompt_davidmiketom = "When David and Mike went to the store, Tom gave a drink to"
# batch_token = tokenizer(prompt_maryjohnjohn, return_tensors="pt", padding=True) # for test
# batch_token = tokenizer(prompt_davidmiketom, return_tensors="pt", padding=True) # for test
prompt_dict_ls_ORIG = [{"text":prompt_maryjohnjohn, "IO_token_id":[6393], "S_token_id":[2516], "S2_token_id":[2516], "END_token_pos":13}]
prompt_dict_ls_NEW = [{"text":prompt_davidmiketom, "IO_token_id":[5119], "S_token_id":[10035], "S2_token_id":[6270], "END_token_pos":13}]
activation_patching(prompt_dict_ls_ORIG, prompt_dict_ls_NEW, model, tokenizer, send_info, recv_info, output_dir, n_layers, n_heads, bsz=1, demo_now=True)

100%|██████████| 1/1 [01:21<00:00, 81.57s/it]
